# NEXUS MCP explorer

Call the NEXUS MCP server's tools **directly, with no LLM in the loop**, and inspect exactly what comes back.

- Run the cells top to bottom. Change any arguments and re-run.
- The connect cell opens a browser for the SBU login if your saved sign-in has expired.
- Every call is appended to `research/calls.jsonl` (not committed), so you can reload and compare results later.
- `alma_renew_user_loan` and `alma_cancel_user_request` change your real library account. The `call()` helper refuses them unless you pass `allow_write=True`.
- **Before committing this notebook, clear its outputs** (Jupyter: *Clear All Outputs*). Results can include your loans, fines and requests.

## 1. Connect

In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT))
from agent import nexus  # same NEXUS settings, login and saved token as the chat agent

LOG = ROOT / "research" / "calls.jsonl"
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 100)

In [ ]:
await nexus.__aenter__()  # opens the browser for the SBU login if needed
client = nexus.client
tools = await client.list_tools()
TOOLS = {t.name: t for t in tools}
print(f"Connected. {len(tools)} tools.")

## 2. What tools are there?

In [ ]:
pd.DataFrame([
    {
        "tool": t.name,
        "required": ", ".join(t.input_schema.get("required") or []),
        "parameters": ", ".join(t.input_schema["properties"]),
        "description": (t.description or "").strip().splitlines()[0],
    }
    for t in tools
])

Full description and input schema for one tool:

In [ ]:
def schema(name):
    """Print a tool's full description and return its input schema."""
    print(TOOLS[name].description)
    return TOOLS[name].input_schema

schema("primo_search")

## 3. Helpers

- `call(tool, **args)`: call any tool, log it, return the parsed result.
- `q("term", ...)`: build a correctly shaped field query, `{"terms": [...], "op": "AND"}`.
- `primo(**args)`: run `primo_search` and return `(table, raw)`: a table of the results **in the order the server returned them**, plus the raw response.
- `compare(a, b)`: compare two result tables (top-1, overlap, rank differences).
- `repeat(n, **args)`: run the same search n times to check whether the server returns the same list every time.

In [ ]:
WRITE_TOOLS = {"alma_renew_user_loan", "alma_cancel_user_request"}


async def call(tool, allow_write=False, **args):
    """Call one NEXUS tool directly (no LLM). Returns the parsed result; logs the call to calls.jsonl."""
    if tool in WRITE_TOOLS and not allow_write:
        raise PermissionError(f"{tool} changes your library account. Pass allow_write=True if you mean it.")
    t0 = time.perf_counter()
    r = await client.call_tool(tool, args, raise_on_error=False)
    seconds = round(time.perf_counter() - t0, 3)
    text = "".join(getattr(c, "text", "") for c in r.content)
    try:
        data = json.loads(text)
    except ValueError:
        data = text
    with LOG.open("a") as f:
        f.write(json.dumps({"at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "tool": tool,
                            "args": args, "error": r.is_error, "seconds": seconds, "result": data}, default=str) + "\n")
    print(f"{tool}: {'ERROR' if r.is_error else 'ok'} in {seconds}s")
    if r.is_error:
        print(text)
    return data


def q(*terms, op="AND"):
    """A field query in the shape primo_search requires: q("Silent Spring") -> {"terms": ["Silent Spring"], "op": "AND"}"""
    return {"terms": list(terms), "op": op}


COLUMNS = ["Record ID", "Title", "Author", "Edition", "Publication Date", "Physical Format",
           "Holding ID", "Available", "Is physical", "Is electronic", "Delivery category"]


def items_table(data):
    """primo_search items as a table, in server order. sbu_copy = has a Holding ID (Stony Brook owns it)."""
    items = data.get("items", []) if isinstance(data, dict) else []
    df = pd.DataFrame(items)
    if df.empty:
        return df
    df = df[[c for c in COLUMNS if c in df.columns]].copy()
    df.insert(0, "rank", range(1, len(df) + 1))
    df["sbu_copy"] = df["Holding ID"].notna() if "Holding ID" in df else False
    return df.set_index("rank")


async def primo(**args):
    """Run primo_search and return (table, raw response). Shows the totals and the equivalent website search."""
    data = await call("primo_search", **args)
    if not isinstance(data, dict):
        return pd.DataFrame(), data
    print(f"total={data.get('total')}  page={data.get('page')} of {data.get('pages')}  items on this page={len(data.get('items', []))}")
    if data.get("search_url"):
        display(Markdown(f"[Same search on the library website]({data['search_url']})"))
    return items_table(data), data


def compare(a, b, k=10, names=("A", "B")):
    """Compare the top k of two result tables by Record ID."""
    ids_a, ids_b = list(a["Record ID"][:k]), list(b["Record ID"][:k])
    both = set(ids_a) & set(ids_b)
    print(f"top-1 same: {bool(ids_a and ids_b and ids_a[0] == ids_b[0])}")
    print(f"overlap in top {k}: {len(both)} of {min(len(ids_a), len(ids_b))}")
    titles = {**dict(zip(b["Record ID"], b["Title"])), **dict(zip(a["Record ID"], a["Title"]))}
    rows = [{"Record ID": rid, "Title": titles.get(rid),
             f"rank in {names[0]}": ids_a.index(rid) + 1 if rid in ids_a else None,
             f"rank in {names[1]}": ids_b.index(rid) + 1 if rid in ids_b else None}
            for rid in dict.fromkeys(ids_a + ids_b)]
    return pd.DataFrame(rows).astype({f"rank in {names[0]}": "Int64", f"rank in {names[1]}": "Int64"})


async def repeat(n=3, **args):
    """Run the same primo_search n times; report whether the ordered result list ever changes."""
    runs = []
    for _ in range(n):
        df, _raw = await primo(**args)
        runs.append(list(df["Record ID"]) if not df.empty else [])
    same = all(r == runs[0] for r in runs)
    print(f"\n{n} runs, identical ordered results every time: {same}")
    return runs

## 4. Searches

### Keyword search

In [ ]:
df, raw = await primo(keywords=q("marine biology"), limit=10)
df

### Title + author (a known item)

In [ ]:
df_ta, raw_ta = await primo(title=q("Silent Spring"), author=q("Carson"), limit=10)
df_ta

### A wrongly shaped call

This is the kind of call gemma4 kept sending: the title as a plain string. The server rejects it, and the error text below is exactly what the model sees.

In [ ]:
await call("primo_search", title="Silent Spring", author="Rachel Carson")

### Filters and sorting

Try `resources_types`, `sort` (`relevance`, `newest`, `oldest`, `title`, `author`), `languages`, `online_resources`, `peer_reviewed`, `page`.

In [ ]:
df_new, raw_new = await primo(keywords=q("CRISPR"), resources_types=["books"], sort="newest", limit=10)
df_new

## 5. Compare two searches

Same book, two ways of asking: keywords versus title and author fields.

In [ ]:
df_kw, _ = await primo(keywords=q("Silent Spring Rachel Carson"), limit=10)
compare(df_kw, df_ta, names=("keywords", "title+author"))

## 6. Is the server consistent?

The same search several times. If the order changes between identical calls, small differences elsewhere are noise.

In [ ]:
runs = await repeat(3, keywords=q("marine biology"), limit=10)

## 7. Look at one record in full

Everything the server returns for a single result, including fields the table leaves out.

In [ ]:
raw_ta["items"][0]

### Facets

Counts for the whole result set, not per item.

In [ ]:
for facet in raw_ta.get("facets", []):
    for name, values in facet.items():
        print(f"{name}: {values[:8]}")

## 8. Other tools

Library information:

In [ ]:
libraries = await call("alma_get_libraries")
libraries

In [ ]:
await call("alma_get_opening_hours", scope="STBMN")  # STBMN = Main Stacks; codes come from alma_get_libraries

Your own account (personal data: clear these outputs before committing):

In [ ]:
await call("alma_get_user_loans", loan_status="Active")

In [ ]:
await call("alma_get_user_fees")

## 9. Reload the call log

Every call made from this notebook, across sessions.

In [ ]:
log = pd.read_json(LOG, lines=True)
log[["at", "tool", "args", "error", "seconds"]].tail(20)

## 10. Disconnect

In [ ]:
await nexus.__aexit__(None, None, None)
print("Disconnected.")